# GLM-5.2 @131k — which modules can be recomputed, which can be offloaded

**LPS-1062 activation placement.** Built 2026-08-20 (turing) from the tree that
actually booted: trainers `jackrao/lps-1062-actplace` @ `1764a816`,
megatron-core 0.19.0 at `basetenlabs/Megatron-LM` @ `2c3d720cf`.

## Why this notebook exists

Arm 3b — selective recompute plus offload of `moe_act` and `attn_proj` — does not
fit at 131k with 2 datums per step. Two attempts OOMed at **258.4 GiB**
torch-allocated on a 267.7 GiB card, *identically* with the backpressure valve
uncapped and capped at 4, so the overshoot is **resident** memory rather than
offload copies still in flight. Roughly 20-30 GiB has to come off the first
stage. The question is therefore: what is left to move, and by which mechanism?

The tempting assumption is that a transformer decomposes into named modules, so
every module has a name, and each name is either recomputable or offloadable.
The first half is true. The second half is where it breaks, and that gap is the
point of the table below.

## The three things worth knowing before reading the table

**1. The vocabularies are hand-curated lists, not the module tree.** There are
eight offloadable names and eight selective-recomputable names, and they overlap
in exactly two (`core_attn`, `moe_act`). Neither list is derived from
`named_modules()`; both are hard-coded sets validated by assertion.

**2. The names denote tensors at call sites, not modules.** `attn_norm` offload
ships the `hidden_states` *entering* the attention norm. `core_attn` offload
ships `query`. `attn_proj` offload ships `core_attn_out`, which is the *input* to
`linear_proj`. So the mapping from module to mechanism is many-to-one in both
directions — `layernorm` recompute covers two different modules, `moe` recompute
covers an entire subtree — and some names attach to a tensor that no single
module owns.

**3. Full recompute is a different mechanism entirely and names nothing.**
`recompute_granularity='full'` takes `recompute_method` (`uniform`/`block`) plus
`recompute_num_layers`: it checkpoints *blocks of whole layers*, stores each
block's input, and re-runs everything inside. It therefore reaches intermediates
that have no name in either vocabulary. That is measured, not asserted — rung 1's
census under full recompute found `attn_proj` resident at **0.000 GiB/set** and
`moe_act` at 0.042 against a 0.75-1.15 eager estimate.

So the useful reading of the table is not "offloadable or not" but **how many rows
are reachable by neither mechanism** — because that is where the missing 20-30 GiB
most likely lives.

## The matrix

One row per module in a GLM-5.2 decoder layer, plus the model-level modules.

- `recompute` — the `recompute_modules` name that covers this module, or `—`.
- `offload` — the `offload_modules` name whose hook site sits on this module's
  saved tensor, or `—`.
- `offload_live_on_glm` — whether that hook actually **fires** on GLM-5.2.
  GLM runs `AbsorbedMLA`, whose file contains exactly one `off_interface` call
  (`attn_proj`). The `qkv_linear` and `core_attn` hooks live in `attention.py` /
  `multi_latent_attention.py`, whose forward does not run on this path — so those
  flags bake to `True` and then never fire. This column is the difference between
  the nominal vocabulary and the one you actually have.

In [ ]:
import pandas as pd

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 46)
pd.set_option('display.width', 200)

# scope: 'layer' = per decoder layer (x78), 'model' = once per pipeline stage.
# Every source reference below was read from the booted tree, not inferred.
ROWS = [
    # ---- layer entry / norms -------------------------------------------------
    dict(scope='layer', module='input_layernorm', kind='norm',
         recompute='layernorm', offload='attn_norm', offload_live_on_glm=True,
         source='recompute transformer_layer.py:450 / offload :617'),
    dict(scope='layer', module='pre_mlp_layernorm', kind='norm',
         recompute='layernorm', offload='mlp_norm', offload_live_on_glm=True,
         source='recompute transformer_layer.py:450 / offload :749'),

    # ---- attention (AbsorbedMLA) --------------------------------------------
    dict(scope='layer', module='self_attention.linear_q_down_proj', kind='linear',
         recompute='—', offload='qkv_linear', offload_live_on_glm=False,
         source='hook attention.py:1347 / mla:342 — AbsorbedMLA fwd bypasses'),
    dict(scope='layer', module='self_attention.linear_kv_down_proj', kind='linear',
         recompute='—', offload='qkv_linear', offload_live_on_glm=False,
         source='same hook as linear_q_down_proj'),
    dict(scope='layer', module='self_attention.linear_q_up_proj', kind='linear',
         recompute='mla_up_proj', offload='—', offload_live_on_glm=None,
         source='absorbed_mla.py:182,659 (CheckpointWithoutOutput)'),
    dict(scope='layer', module='self_attention.linear_kv_up_proj', kind='linear',
         recompute='mla_up_proj', offload='—', offload_live_on_glm=None,
         source='absorbed_mla.py:182,659'),
    dict(scope='layer', module='self_attention.q_layernorm', kind='norm',
         recompute='—', offload='—', offload_live_on_glm=None,
         source='no hook in either vocabulary'),
    dict(scope='layer', module='self_attention.kv_layernorm', kind='norm',
         recompute='—', offload='—', offload_live_on_glm=None,
         source='no hook in either vocabulary'),
    dict(scope='layer', module='self_attention.core_attention', kind='attention',
         recompute='core_attn', offload='core_attn', offload_live_on_glm=False,
         source='recompute absorbed_mla.py:845 / offload hook mla:380 bypassed'),
    dict(scope='layer', module='self_attention.linear_proj', kind='linear',
         recompute='—', offload='attn_proj', offload_live_on_glm=True,
         source='absorbed_mla.py:925 (ships core_attn_out, its input)'),
    dict(scope='layer', module='self_attn_bda', kind='fused bias-dropout-add',
         recompute='—', offload='—', offload_live_on_glm=None,
         source='no hook in either vocabulary'),
    dict(scope='layer', module='dsa_indexer (GLM-5.2 DSA top-k)', kind='sparse-attn index',
         recompute='—', offload='—', offload_live_on_glm=None,
         source='GLM-specific; no hook in either vocabulary'),

    # ---- MoE MLP (75 of 78 layers) ------------------------------------------
    dict(scope='layer', module='mlp [MoE layer]', kind='moe container',
         recompute='moe', offload='—', offload_live_on_glm=None,
         source='transformer_layer.py:1508 (whole-subtree checkpoint)'),
    dict(scope='layer', module='mlp.router', kind='router',
         recompute='(via moe)', offload='—', offload_live_on_glm=None,
         source='no name of its own in either vocabulary'),
    dict(scope='layer', module='mlp.token_dispatcher (permute/unpermute, a2a)',
         kind='dispatch',
         recompute='(via moe)', offload='—', offload_live_on_glm=None,
         source='no name of its own in either vocabulary'),
    dict(scope='layer', module='mlp.experts.linear_fc1', kind='grouped linear',
         recompute='(via moe)', offload='expert_fc1', offload_live_on_glm=True,
         source='experts.py:701 (ships permuted_local_hidden_states)'),
    dict(scope='layer', module='mlp.experts activation_func', kind='activation',
         recompute='moe_act', offload='moe_act', offload_live_on_glm=True,
         source='recompute experts.py:261 / offload :714 (ships fc1_output)'),
    dict(scope='layer', module='mlp.experts.linear_fc2', kind='grouped linear',
         recompute='(via moe)', offload='—', offload_live_on_glm=None,
         source='no name of its own in either vocabulary'),
    dict(scope='layer', module='mlp.shared_experts', kind='dense expert',
         recompute='shared_experts', offload='—', offload_live_on_glm=None,
         source='shared_experts.py:153, moe_layer.py:253'),
    dict(scope='layer', module='mlp_bda', kind='fused bias-dropout-add',
         recompute='—', offload='—', offload_live_on_glm=None,
         source='no hook in either vocabulary'),

    # ---- dense MLP (3 of 78 layers, all on stage 0) -------------------------
    dict(scope='layer', module='mlp [dense MLP] (3 layers)', kind='dense mlp',
         recompute='mlp', offload='—', offload_live_on_glm=None,
         source='transformer_layer.py:512'),
    dict(scope='layer', module='mlp.linear_fc1 / activation_func / linear_fc2 (dense)',
         kind='dense mlp internals',
         recompute='(via mlp)', offload='—', offload_live_on_glm=None,
         source='fused_group_mlp exists (experts.py:616) but trainer rejects it'),

    # ---- model level ---------------------------------------------------------
    dict(scope='model', module='embedding', kind='embedding',
         recompute='—', offload='—', offload_live_on_glm=None,
         source='no hook in either vocabulary'),
    dict(scope='model', module='final_layernorm', kind='norm',
         recompute='—', offload='—', offload_live_on_glm=None,
         source='no hook in either vocabulary'),
    dict(scope='model', module='output_layer / chunked LM head', kind='logits + loss',
         recompute='—', offload='—', offload_live_on_glm=None,
         source='MEASURED 26.04 GiB on the last stage (rung 1)'),
]

df = pd.DataFrame(ROWS)
df

## The gap: rows reachable by neither mechanism

This is the answer to "is every module either offloadable or recomputable?" —
no, and the ones that are neither are not a rounding error.

In [ ]:
neither = df[(df.recompute.isin(['—'])) &
             ((df.offload == '—') | (df.offload_live_on_glm == False))]
print(f'{len(neither)} of {len(df)} rows are reachable by NEITHER mechanism on GLM-5.2\n')
neither[['scope', 'module', 'kind', 'source']]

In [ ]:
# What is actually available as a lever, versus already in use.
OFFLOAD_VOCAB = ['attn_norm', 'qkv_linear', 'core_attn', 'attn_proj',
                 'mlp_norm', 'expert_fc1', 'moe_act', 'fused_group_mlp']
RECOMPUTE_VOCAB = ['core_attn', 'moe_act', 'layernorm', 'mla_up_proj',
                   'mlp', 'moe', 'shared_experts', 'gdn_norm_out']

IN_USE_OFFLOAD = ['moe_act', 'attn_proj']          # arm 3b
IN_USE_RECOMPUTE = ['core_attn']                   # recompute_modules default

# Why each unavailable name is unavailable on THIS model.
BLOCKED_OFFLOAD = {
    'qkv_linear': 'hook bypassed by AbsorbedMLA; also shares its tensor with the '
                  'core-attn checkpoint (double-store or H2D in the critical path)',
    'core_attn': 'hook bypassed by AbsorbedMLA; phase 2 at best by design',
    'fused_group_mlp': 'trainer validator rejects it (needs the TE op fuser)',
}
BLOCKED_RECOMPUTE = {
    'gdn_norm_out': 'GatedDeltaNet only; GLM-5.2 has none',
}

print('OFFLOAD')
for n in OFFLOAD_VOCAB:
    state = ('IN USE' if n in IN_USE_OFFLOAD
             else f'blocked - {BLOCKED_OFFLOAD[n]}' if n in BLOCKED_OFFLOAD
             else 'AVAILABLE')
    print(f'  {n:<17} {state}')

print('\nSELECTIVE RECOMPUTE')
for n in RECOMPUTE_VOCAB:
    state = ('IN USE' if n in IN_USE_RECOMPUTE
             else f'blocked - {BLOCKED_RECOMPUTE[n]}' if n in BLOCKED_RECOMPUTE
             else 'AVAILABLE')
    print(f'  {n:<17} {state}')

## What this means for the 20-30 GiB

**Three offload levers are untried:** `attn_norm`, `mlp_norm`, `expert_fc1`.
Every other offload name is either already on, bypassed by `AbsorbedMLA`, or
rejected by our own validator. So the offload surface on this model is five
names, not eight, and two are spent.

**Five recompute levers are untried:** `layernorm`, `mla_up_proj`, `moe`, `mlp`,
`shared_experts`. We are running the `recompute_modules` **default**, which is
`["core_attn"]` — nothing else. This is the larger and cheaper surface, and it
is the one nobody has touched.

**`layernorm` recompute is the lever I would reach for first.** It covers both
`input_layernorm` and `pre_mlp_layernorm` across all 78 layers, it uses
output-discarding checkpointing (cheap — a couple of kernels), and unlike the
`attn_norm`/`mlp_norm` offloads it consumes **no PCIe bandwidth at all**. That
matters because bandwidth is already the binding constraint on the offload
design: the requirement was 22-24 GB/s per GPU against 27.5 measured
device-to-host when NUMA-local, and the pinned pool for just `moe_act` +
`attn_proj` already reached ~17.5 GiB per rank.

**`mla_up_proj` recompute is the second.** It is the only mechanism of any kind
that reaches the MLA up-projection tensors, and it does fire on `AbsorbedMLA`
(`absorbed_mla.py:182`) — unlike the attention-side *offload* hooks.

### Two caveats before either is spent on a boot

1. **fp8 interaction.** `layernorm` and `moe_act` recompute are rejected outright
   under fp8 with `delayed` scaling, and otherwise require
   transformer-engine >= 2.6.0dev0 (`transformer_config.py:1786-1797`). I found no
   explicit `fp8` / `fp8_recipe` assignment in the trainer's config mapping, which
   suggests `config.fp8` is `None` and the restriction does not bite — but
   "GLM-5.2-**FP8**" is an fp8 checkpoint and `absorbed_mla.py:662` passes a
   `quantization` flag into `CheckpointWithoutOutput`, so this must be confirmed
   from the resolved config at boot rather than assumed.
2. **The rows that are reachable by nothing.** `q_layernorm`, `kv_layernorm`, the
   two bda ops, the router, the token dispatcher's permutation state, the DSA
   indexer, and the 26.04 GiB LM head have no name in either vocabulary. If the
   resident excess is concentrated there, **no combination of the settings in this
   table fixes it**, and the honest options become the block+K recompute dial, a
   new hook (a real code change), or fewer in-flight microbatches.

### Which is why the composition measurement is still the gating unknown

Nothing above says *where* the ~113.7 GiB above the full-recompute baseline
actually sits. The plan's inherited glue band (0.64-1.14 GiB per MoE set) and the
measured lower bound (>=0.78 GiB/set at 140 sets) are consistent with it being
mostly norms, residuals and router bookkeeping — i.e. concentrated in exactly the
rows that no name reaches. That is a guess, not a measurement. It needs a 3b run
that survives to a plateau and dumps allocator snapshots, and then
`memory_census.py` for per-module attribution. Rung 1 could not measure it,
because under full recompute those tensors are not resident to measure.

## Provenance

| claim | source |
|---|---|
| offload vocabulary (8 names) | `transformer_config.py:1817-1829`; mirrored in `ActivationOffloadConfig`, `loops_models/control.py` |
| recompute vocabulary (8 names) + default `["core_attn"]` | `transformer_config.py:536-551`, `:1727-1745` |
| full recompute is layer/block granularity | `transformer_config.py:508-529` (`recompute_method`, `recompute_num_layers`) |
| offload hook sites | `transformer_layer.py:617,749`; `attention.py:1347,1504,1577`; `multi_latent_attention.py:342,380,461`; `absorbed_mla.py:925`; `experts.py:616,701,714` |
| recompute hook sites | `transformer_layer.py:450,512,1508`; `absorbed_mla.py:182,845`; `multi_latent_attention.py:172`; `shared_experts.py:153`; `moe_layer.py:253`; `experts.py:261` |
| AbsorbedMLA has only the `attn_proj` offload hook | `absorbed_mla.py` — single `off_interface` call, at :925 |
| `attn_norm`/`mlp_norm` guards | `transformer_layer.py:1373-1424` — require a non-`IdentityOp` norm; the MoE-layer restrictions are gated on CUDA graphs, which we do not use |
| 78 layers, 3 dense + 75 sparse, all dense on stage 0 | checkpoint config `ba978f7d`, rung 1 report §7 |
| LM head 26.04 GiB on the last stage | rung 1 census, stable across all 8 last-stage ranks |
| 258.4 GiB OOM, identical capped and uncapped | `RUNG3_3B_REPORT.md` §0, §5 |

**Not verified:** the instantiated module tree was read from the submodule
dataclasses and hook sites, not dumped from a live model — the trainer is down.
GLM may use the fused `linear_qkv_down_proj` rather than split q/kv down-projections.
One `named_modules()` call on the next boot settles it.